# Canonical Data Validation and Analytical Readiness

## Infocreon Phase 3 — Post #2

This notebook validates and profiles the existing canonical dataset:
`data/canonical/intelligence_data.csv`.

It reviews structural quality, coverage, dataset archetype, and analytical readiness. It does not create a new dataset, develop predictive models, or modify the operational dashboard.

## 1. Scope and Source

The canonical CSV is the sole analytical source. Generated JSON assessments are used as evidence for reporting; they do not replace the canonical data.

The dataset is controlled synthetic data. Structural validity does not establish real-world accuracy or representativeness.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()

# Allow execution when the notebook kernel starts in the repository
# root or in the notebooks directory.
if not (ROOT / "data" / "canonical" / "intelligence_data.csv").exists():
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "canonical" / "intelligence_data.csv"
OUTPUT_DIR = ROOT / "data-science" / "outputs"

assert DATA_FILE.exists(), f"Canonical CSV not found: {DATA_FILE}"

df = pd.read_csv(DATA_FILE, low_memory=False)

print("Source:", DATA_FILE.relative_to(ROOT))
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Columns match expected count:", len(df.columns) == 20)

In [ ]:
expected_columns = [
    "record_id", "record_type", "observed_at", "entity_id",
    "related_entity_id", "entity_name", "category", "subcategory",
    "status", "stage", "metric_name", "metric_value", "metric_unit",
    "text_value", "latitude", "longitude", "source_name",
    "source_record_id", "is_synthetic", "data_version",
]

print("Missing expected columns:",
      [c for c in expected_columns if c not in df.columns])
print("Unexpected columns:",
      [c for c in df.columns if c not in expected_columns])
print("Column order matches:",
      list(df.columns) == expected_columns)

assert set(expected_columns).issubset(df.columns)

In [ ]:
profile_path = OUTPUT_DIR / "canonical_profile.json"
quality_path = OUTPUT_DIR / "quality_assessment.json"
representativeness_path = OUTPUT_DIR / "representativeness_assessment.json"
readiness_path = OUTPUT_DIR / "analytical_readiness.json"

required_outputs = [
    profile_path,
    quality_path,
    representativeness_path,
    readiness_path,
]

for path in required_outputs:
    assert path.exists(), f"Missing assessment output: {path}"

profile = json.loads(profile_path.read_text(encoding="utf-8"))
quality = json.loads(quality_path.read_text(encoding="utf-8"))
representativeness = json.loads(
    representativeness_path.read_text(encoding="utf-8")
)
readiness = json.loads(readiness_path.read_text(encoding="utf-8"))

print("Profile rows:", profile["row_count"])
print("Quality status:", quality["overall_structural_status"])
print("Representativeness checks:",
      representativeness["structural_checks"])
print("Final conclusion:", readiness["final_conclusion"])

In [ ]:
dates = pd.to_datetime(df["observed_at"], errors="coerce", utc=True)

summary = {
    "record_count": len(df),
    "column_count": len(df.columns),
    "unique_cities": int(df["entity_name"].nunique(dropna=True)),
    "unique_pollutants": int(df["subcategory"].nunique(dropna=True)),
    "unique_timestamps": int(dates.nunique(dropna=True)),
    "unique_calendar_dates": int(
        dates.dropna().dt.strftime("%Y-%m-%d").nunique()
    ),
    "valid_coordinate_rows": int(
        pd.to_numeric(df["latitude"], errors="coerce").between(-90, 90)
        .__and__(
            pd.to_numeric(df["longitude"], errors="coerce").between(-180, 180)
        ).sum()
    ),
}

pd.Series(summary, name="Observed value").to_frame()

## 2. Interpretation

The current profile shows three unique timestamps on one calendar date. This is sparse repeated measurement coverage, not evidence of long-term temporal trends.

The dataset's eight city names and four pollutant categories support consideration of bounded descriptive comparisons. Because the data is synthetic, these comparisons must not be represented as verified real-world findings.

## 3. Predictive Intelligence Gate

Predictive Intelligence is rejected for this phase. A model must not be developed without a defined and validated target, sufficient history, documented decision-time inputs, leakage assessment, target-distribution analysis, a validation plan, and a genuine business need.

## 4. Conclusion

The current assessment conclusion is read from `analytical_readiness.json`. Any analytical work must follow the documented track limitations and use the existing canonical CSV as its source.